# VEP-nAChR — the story so far

*A memory-restore notebook. Read top-to-bottom and you're caught up. Every number is the headline macro-F1 (3-class: **LOF / no-net-effect / GOF**), usually from 10-seed runs unless noted.*

## 1. The goal (why this project exists)

Most variant predictors (AlphaMissense, CADD, PolyPhen) answer one question: *"is this mutation damaging?"* We answer a harder one: *"does this mutation make the receptor **gain** function or **lose** function?"* A damaging variant can be either — those tools can't tell the two apart.

We do this specifically for **nAChR** (nicotinic acetylcholine receptor) subunits. No existing tool covers nAChR (MissION covers 47 *voltage-gated* genes, not nAChR), so this is a genuine gap. Our angle: the **first nAChR-specific GOF/LOF direction predictor**.

## 2. The data

| species | variants |
|---|---:|
| human | 542 |
| rat | 174 |
| mouse | 80 |
| **total** | **797** |

- **16 genes** (CHRNA1-7, A9, A10, CHRNB1-4, CHRND, CHRNE, CHRNG).
- **3 labels**, class-imbalanced (LOF most common, then NNE, then GOF).
- Each variant → **~75 features**: 24 physicochemical, 3 substitution, 20 positional, 3 conservation, 7 structural, 2 ESM-2 effect, 16 ESM-2-embedding (PCA).
- Evaluation metric: **macro-F1**, never plain accuracy (imbalance).

## 3. The two phases (the roadmap)

**Phase 1 — "the biology chase"** (Runs VIII–XXV). We kept feeding the model *better biology*: conservation, bigger language models, structure, conformational state, coevolution, dynamics. Almost everything came back **null**, and performance froze at **~0.50 macro-F1**. Painful, but it taught us where the signal is *not*.

**Phase 2 — "the gene-aware reframe"** (Runs XXVI–XXXVIII). We stopped forcing the model to be gene-blind and gave it **gene identity** as an input. Performance jumped, and the δ/γ/ε story emerged.

## 4. Phase 1 — the null streak (one lesson, ~12 hypotheses)

Everything below ran under **gene-grouped CV** (leave-one-gene-out). The common theme: *more biological detail ≠ more predictive signal*, and the whole thing plateaus at ~0.50.

| what we tried | result |
|---|---|
| Conservation + bigger ESM-2 (VIII, IX) | +0.023 (3-class), ~0 (binary); 650M ≈ 150M |
| Embeddings / stacking / ordinal (XIII) | all null/negative; CatBoost 0.490 baseline |
| Mutation-aware structural extractors (XV) | net-negative; reverted |
| **α7 two-state (open↔closed) conformational** (XVI–XIX) | null, even in homomeric α7; ΔΔG two-state null too |
| Coevolution, embedding deltas, functional regions, allosteric networks (XX–XXV) | all null/negative |
| Label audit (XXIII) | NNE labels 95.9% single-source / 40% AM-pathogenic, but dropping them only moves F1 −0.007 (noise, orthogonal to direction) |
| Within-family CV (XXIV) | strictly worse than pooled → "0.50 is a real ceiling" |

**Lesson:** the direction signal is *not* sitting in structure or conservation alone. The ~0.50 ceiling was real — but only for a **gene-agnostic** model.

## 5. Phase 2 — the turning point (Run XXVI)

The reframe: **"which gene is it?" is legitimate information**, not a confound. In real deployment a clinician *knows* the gene, so there's no reason to hide it from the model.

Feeding 16 gene-identity columns as input: **0.50 → 0.63 macro-F1**.

This changed the protocol:
- **Headline** = gene-aware (`cv_mode="standard"`).
- **Floor** = leave-one-gene-out (kept as a sanity bound, not the headline).

## 6. Phase 2 — the wins that survived

| run | finding |
|---|---|
| XXVII | GNM **gating** + coupling×mutation interaction **positive** under gene-aware (+0.013/+0.019) — first structural win; the 12 nulls were a CV artifact |
| XXIX | mouse/rat augmentation **+0.05** over human-only (mostly NNE/GOF label rebalancing, not species diversity) |
| XXX | not at the data ceiling: halving data costs ~0.06–0.09; minority GOF/NNE worth more per point |
| XXXI–XXXIV | **δ/γ/ε alone beats the full model** (0.69 vs 0.65), *biological not identity-confounded*, p ≤ 0.001; δγε leans on ESM-2+gating, hurt by physicochemical |
| XXXV | a 7-feat functional (GO) encoding can *substitute* for identity, but only weakly augments it |
| XXXVIII | δγε 0.70 is **not** majority-class cheating (always-LOF = 0.259; GOF & NNE F1 both > 0.52) |

**Current headline:** full model ~0.645 macro-F1; δ/γ/ε subset ~0.69–0.70 (best).

## 7. Cross-species mapping (how mouse/rat became human)

This is the part that *does* work, and it works for a precise reason. Three levels:

**Level 0 — genes don't need mapping.** Gene symbols are identical across human/mouse/rat (`CHRNA1 == Chrna1`). Same gene, same job.

**Level 1 — fix the numbering frame** (`scripts/normalize_positions.py`). Papers report positions in different frames: some count the **signal peptide** (precursor), others strip it (mature protein), and a few are typos. We anchor every row back to precursor numbering using the one thing papers report reliably — the **wild-type amino acid** at the reported position. (mature = +signal-peptide length; typo = unique match within ±5; isoform = within +15..+40.)

**Level 2 — residue-level ortholog alignment** (`scripts/map_ortholog_positions.py`). We take the human canonical RefSeq isoform and its mouse/rat ortholog, **globally align them (BLOSUM62)**, and emit a per-residue map `mouse_pos → human_pos`. Insertions (no human equivalent) are dropped.

**Why this is safe:** a mouse CHRNA7 is ~93% identical to human CHRNA7 — *it's the same protein from a different animal*. At 93% identity the alignment is nearly gap-free, so "position 100 in mouse" = "position 100 in human" almost always. This is only valid in that high-identity regime (see next section).

Identity (human ↔ mouse/rat) ranges **~81–94%** across the 16 genes — see `data/raw/reference_sequences/mapping/gene_summary.csv`.

## 8. The open brainstorm — "map all genes onto one gene" — and why it fails

The idea: pick the *one* gene "most identical to all the others" and map every variant onto it, so we have one big reference. Here are the actual numbers (pairwise BLOSUM62 identity across the 16 human subunits):

| regime | identity |
|---|---|
| human ↔ mouse/rat (same gene) | **81–94%** |
| human gene ↔ other human gene | **27–63%** (mean ~37%) |

The "most central" gene is **CHRNA2** — but it is only **~42% identical** to the other 15 on average, and **32.5%** to its worst match. There is no gene close to all of them.

**Why that kills the idea, simply:** position-mapping *is* alignment, and alignment is only meaningful when two sequences are nearly the same protein. At 93% identity (mouse↔human) you get a near 1:1 map. At 30–60% identity the alignment has many gaps and — more importantly — **"aligned position" no longer means "same functional place."** The residue that is the ligand-binding pocket in one subunit can be a transmembrane segment in another.

**Analogy:** species mapping is translating page 100 of the *UK edition* of a book to page 100 of the *US edition* — basically the same page. Gene mapping is translating page 100 of a *cookbook* to page 100 of a *novel* — both are "page 100," but the content is unrelated.

**We already saw this bite us, empirically:** we homology-mapped structure features onto another subunit's chain at ~47% identity and got *wrong-residue* features — which is why 7 genes without their own structure use their own AlphaFold model (offset 0) instead (see `VEP Nachr2/CLAUDE.md` → PDB mapping). Cross-gene mapping at this identity is a known dead end in this project.

## 9. Where we are / what's next

- The **gene-aware** reframe is the project's current identity: a *gene-conditioned* GOF/LOF predictor.
- The **δ/γ/ε** subunits carry a disproportionately strong signal — worth understanding *why* (biology), not just exploiting.
- Open threads: cross-species augmentation is real but mostly a label-rebalancing effect; the one-gene mapping idea is settled **against** (Section 8).
- Paper target: PSB 2027 position paper (see `ASW/`, `assignments/`).

## 10. Quick reference — key files

- `vep_nachr2/` — the actual library (config, data loader, feature extractors, training).
- `scripts/normalize_positions.py` — numbering-frame fix (human + species → precursor numbering).
- `scripts/map_ortholog_positions.py` — builds the mouse/rat → human residue maps.
- `data/raw/reference_sequences/mapping/gene_summary.csv` — cross-species identity table.
- `data/raw/final.xlsx` → `final_mapped.xlsx` — raw vs position-corrected variants.
- `results.ipynb` — the results tables (markdown-rendered).
- Memory (across sessions): `memory/MEMORY.md` at repo root holds the per-run notes this notebook summarizes.